# Detección de noticias falsas en español — línea base clásica y RoBERTa-BNE

**Proyecto Integrador · Deep Learning**
Referencia: Blanco-Fernández, Otero-Vizoso, Gil-Solla y García-Duque (2024),
*Enhancing Misinformation Detection in Spanish Language with Deep Learning: BERT and RoBERTa
Transformer Models*, Applied Sciences 14(21): 9729.

---

## Qué hace este notebook

Un solo flujo, una sola partición, dos modelos comparables entre sí sobre exactamente los mismos datos:

1. **Línea base clásica** — TF-IDF con regresión logística, SVM lineal y Naive Bayes.
2. **Modelo profundo** — ajuste fino completo de RoBERTa-BNE (`roberta-base-bne`, Barcelona
   Supercomputing Center) con cabeza de clasificación binaria.

Ambos se entrenan sobre el mismo `train`, se ajustan sobre el mismo `val` y se evalúan una única
vez sobre el mismo `test`. La partición se construye una sola vez, en memoria, y ambos modelos la
heredan: no hay archivos intermedios que sincronizar ni riesgo de comparar sobre datos distintos.

## Qué corrige respecto de la versión anterior

| Problema | Corrección |
|---|---|
| Dos notebooks pegados, con `CONFIG`, `fijar_semillas` y `cargar_y_limpiar` duplicados y redefinidos a mitad de camino | Un único flujo con una sola configuración y una sola función por responsabilidad |
| Hiperparámetros del artículo (46.000 ejemplos) aplicados tal cual a un corpus de 793 | Escalado explícito y declarado cuando el corpus es pequeño |
| Entrenamiento colapsado: el modelo predecía una sola clase y la parada temprana lo daba por bueno | Detección de colapso, mínimo de épocas antes de parar, criterio de selección con desempate por MCC y reintento automático con tasa de aprendizaje mayor |
| Una sola semilla, un solo número, sin dispersión | Varias semillas, se reporta media y desviación estándar |
| `n_test = 80` reportado como cifra puntual | Intervalo de confianza por bootstrap y prueba de McNemar entre modelos |
| Ruta del archivo ignorada al construir el dataset (`ruta_csv` en lugar de la ruta efectiva) | Una sola ruta, resuelta y verificada al inicio |
| Expresión regular de limpieza mal escapada, nunca coincidía | Corregida y con prueba en la propia celda |
| Generador de datos sintéticos duplicado, prohibido por la guía de la asignatura | Eliminado: si falta el archivo, el notebook se detiene |
| Padding a longitud fija de 128 para todas las noticias (mediana real: 49 tokens) | Padding dinámico por lote |

## Requisitos

- Un archivo `.xlsx` o `.csv` con las columnas de etiqueta, titular, descripción y fecha.
- GPU para la parte profunda (la clásica corre en CPU en segundos).

---
## §1 · Entorno

Colab ya trae `torch`, `pandas`, `numpy`, `scikit-learn`, `scipy` y `matplotlib`. Solo se asegura
una versión reciente de `transformers`. Si la celda instala algo, hay que reiniciar el entorno de
ejecución antes de continuar.

In [ ]:
!pip install -q "transformers>=4.44" "huggingface_hub>=0.24"

In [ ]:
import os
import re
import json
import time
import random
import hashlib
import platform
import unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix, classification_report,
)

import torch
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import (
    AutoTokenizer, AutoConfig,
    AutoModelForSequenceClassification, AutoModelForMaskedLM,
    get_linear_schedule_with_warmup,
)

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"python       : {platform.python_version()}")
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"dispositivo  : {DISPOSITIVO}"
      + (f" ({torch.cuda.get_device_name(0)})" if DISPOSITIVO.type == "cuda" else ""))
if DISPOSITIVO.type != "cuda":
    print("AVISO: sin GPU el ajuste fino de RoBERTa es inviable en tiempo razonable.")

---
## §2 · Configuración

**Es la única celda que hay que modificar.** Todo lo demás se deriva de aquí y queda registrado en
`configuracion.json` al final.

### Nombres de columna

No se fija un nombre único: se declara una lista de alias por campo y el notebook resuelve el
primero que exista en el archivo. Así el mismo notebook sirve para el corpus español del artículo
(`Titulo`, `Descripcion`, `Fecha`) y para el colombiano (`title`, `description`, `date`) sin tocar nada.

### Hiperparámetros

`hp_articulo` reproduce la Tabla 3 del artículo, columna RoBERTa. Esos valores fueron ajustados
sobre 46.000 ejemplos de entrenamiento. Con un corpus de unos cientos de noticias, 10 épocas a
`1e-5` son unos 400 pasos de optimización: insuficientes para que la cabeza de clasificación
—inicializada al azar— llegue a separar nada, y el resultado es un modelo que predice siempre la
clase mayoritaria.

Por eso, cuando `n_train` queda por debajo de `umbral_corpus_pequeno`, se aplican los valores de
`hp_corpus_pequeno`: más épocas, tasa de aprendizaje mayor y más paciencia. El cambio se imprime
en pantalla y se registra, no ocurre en silencio. Para replicar el artículo al pie de la letra basta
con poner `escalar_si_corpus_pequeno = False`.

### Semillas

`semillas` acepta una lista. Con varias, cada una entrena un modelo completo y al final se reporta
media y desviación estándar. Es lo que permite decir si una diferencia entre modelos es real o
ruido de inicialización, y es justamente lo que el artículo no hace.

In [ ]:
CONFIG = {
    # ---------- DATOS ----------
    "ruta_datos": "/content/dataset_politica_colombiana.xlsx",   # .xlsx o .csv

    # Alias aceptados para cada campo. Se usa el primero que exista en el archivo.
    "alias_columnas": {
        "etiqueta":    ["label", "Label", "etiqueta", "Etiqueta", "clase"],
        "titulo":      ["title", "Titulo", "titulo", "Título", "headline", "titular"],
        "descripcion": ["description", "Descripcion", "descripcion", "Descripción", "body", "cuerpo"],
        "fecha":       ["date", "Fecha", "fecha", "published_at"],
    },
    "campos_texto": ["titulo", "descripcion"],   # lo que ve el modelo, en este orden
    "etiquetas": {0: "falsa", 1: "real"},        # convención del artículo
    "fecha_dia_primero": False,                  # True si las fechas son DD/MM/YYYY
    "anios_validos": [2000, 2026],               # fuera de este rango se marca como fecha imposible
    "submuestra": None,                          # int para pruebas rápidas; None usa todo

    # ---------- PARTICIÓN ----------
    "estrategia_particion": "grupo",             # "grupo" | "temporal"
    "proporciones": None,                        # None = automático según tamaño del corpus
    "umbral_corpus_pequeno": 5000,               # por debajo: 70/15/15 y escalado de hiperparámetros
    "agrupar_variantes": True,                   # evita fuga entre pares real/falsa derivados

    # ---------- LÍNEA BASE CLÁSICA ----------
    "clasico_ngramas": [(1, 1), (1, 2)],
    "clasico_min_df": [2, 5],
    "clasico_C": [0.1, 1.0, 10.0],
    "clasico_max_features": 300_000,

    # ---------- MODELO PROFUNDO ----------
    "modelo_candidatos": [
        "PlanTL-GOB-ES/roberta-base-bne",   # repositorio del artículo (hoy vaciado)
        "BSC-LT/roberta-base-bne",
        "BSC-TeMU/roberta-base-bne",
        "IsGarrido/roberta-base-bne",       # espejo comunitario
    ],
    "modelo_respaldo": "bertin-project/bertin-roberta-base-spanish",
    "verificar_arquitectura": True,
    "verificar_pesos_con_mlm": True,        # comprueba que los pesos preentrenados son reales

    "hp_articulo": {
        "learning_rate": 1e-5, "epocas": 10, "batch_size": 16, "max_seq_length": 128,
        "dropout": 0.15, "weight_decay": 0.001, "warmup_ratio": 0.06,
        "paciencia": 3, "epocas_minimas": 1,
    },
    "hp_corpus_pequeno": {
        "learning_rate": 2e-5, "epocas": 20, "warmup_ratio": 0.10,
        "paciencia": 6, "epocas_minimas": 6, "dropout": 0.10,
    },
    "escalar_si_corpus_pequeno": True,
    "max_grad_norm": 1.0,
    "adam_epsilon": 1e-8,
    "usar_amp": True,
    "usar_pesos_clase": False,
    "reintentar_si_colapsa": True,          # reentrena con lr x3 si el modelo predice una sola clase

    # ---------- PROTOCOLO ----------
    "semillas": [16, 17, 18],               # varias corridas -> media y desviación estándar
    "metrica_seleccion": "f1_macro",
    "bootstrap_n": 2000,                    # remuestreos para los intervalos de confianza

    # ---------- SALIDA ----------
    "dir_salida": "resultados",
    "guardar_modelo": True,
}

SEMILLA_BASE = CONFIG["semillas"][0]
DIR_SALIDA = CONFIG["dir_salida"]
os.makedirs(DIR_SALIDA, exist_ok=True)
CONFIG["usar_amp"] = CONFIG["usar_amp"] and DISPOSITIVO.type == "cuda"

print(f"Datos    : {CONFIG['ruta_datos']}")
print(f"Salidas  : ./{DIR_SALIDA}/")
print(f"Semillas : {CONFIG['semillas']}")

---
## §3 · Utilidades

Funciones sin lógica de negocio que se usan en todo el notebook: fijar semillas, normalizar texto,
imprimir encabezados y calcular el hash del archivo de datos.

`fijar_semillas` cubre las cuatro fuentes de aleatoriedad —`random`, NumPy, PyTorch en CPU y
PyTorch en GPU— y desactiva la selección automática de algoritmos de cuDNN. Aun así, en GPU quedan
diferencias mínimas por acumulación en punto flotante: por eso el protocolo real de reproducibilidad
no es una semilla, sino **varias semillas y una desviación estándar**.

In [ ]:
def fijar_semillas(semilla: int) -> None:
    """Fija las cuatro fuentes de aleatoriedad y desactiva el autotuning de cuDNN."""
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)
    os.environ["PYTHONHASHSEED"] = str(semilla)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def normalizar(texto: str) -> str:
    """Minúsculas, sin tildes, sin puntuación. Para comparar variantes de una misma noticia."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    t = re.sub(r"[^a-z0-9 ]", " ", t.lower())
    return re.sub(r"\s+", " ", t).strip()


def seccion(titulo: str, ancho: int = 78) -> None:
    print("=" * ancho)
    print(titulo)
    print("=" * ancho)


def md5_archivo(ruta: str, bloque: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for trozo in iter(lambda: f.read(bloque), b""):
            h.update(trozo)
    return h.hexdigest()


def ruta(*partes) -> str:
    return os.path.join(DIR_SALIDA, *partes)


fijar_semillas(SEMILLA_BASE)
print(f"Semillas fijadas en {SEMILLA_BASE}.")
print("Prueba de 'normalizar':", repr(normalizar("El Ministro, ¿acusó al Fiscal?")))

---
## §4 · Carga, limpieza y control de calidad

Orden de las operaciones, y el motivo de cada una:

1. **Resolver el archivo y las columnas.** Si no existe el archivo, o falta un campo obligatorio,
   el notebook se detiene aquí. Es preferible a producir resultados sobre datos equivocados.
2. **Construir el campo `texto`** concatenando titular y descripción. La limpieza de puntos dobles
   se aplica con una expresión regular correctamente escapada (en la versión anterior estaba doblemente
   escapada y no coincidía nunca).
3. **Descartar filas sin texto o con etiqueta fuera de {0, 1}.**
4. **Parsear fechas** y marcar las que caen fuera de `anios_validos`. Un corpus con fechas futuras
   invalida la partición temporal, así que conviene saberlo antes y no después.
5. **Eliminar duplicados** por texto normalizado, no por coincidencia exacta de cadena: dos filas que
   solo difieren en un espacio o una tilde son la misma noticia.
6. **Detectar contradicciones**: mismo texto normalizado con etiquetas opuestas. Son ruido de
   etiquetado y se eliminan por completo, porque ningún modelo puede acertar en ambas y contaminan
   el conjunto de prueba.
7. **Asignar `row_id` propio.** El identificador que trae el corpus del artículo es inservible: la
   columna `ID` contiene el literal `"ID"` en las 57.231 filas.

In [ ]:
def resolver_columnas(df: pd.DataFrame, alias: dict) -> dict:
    """Devuelve {campo: nombre real de la columna}. Falla si falta un campo obligatorio."""
    resuelto, faltantes = {}, []
    for campo, candidatos in alias.items():
        encontrado = next((c for c in candidatos if c in df.columns), None)
        if encontrado is None:
            faltantes.append(campo)
        else:
            resuelto[campo] = encontrado
    obligatorios = {"etiqueta", "titulo", "descripcion"}
    if obligatorios & set(faltantes):
        raise ValueError(
            f"Faltan columnas obligatorias {sorted(obligatorios & set(faltantes))}. "
            f"El archivo tiene: {df.columns.tolist()}"
        )
    if "fecha" in faltantes:
        print("AVISO: no hay columna de fecha. La partición temporal no estará disponible.")
    return resuelto


def leer_tabla(ruta_archivo: str) -> pd.DataFrame:
    if not os.path.exists(ruta_archivo):
        raise FileNotFoundError(
            f"No existe '{ruta_archivo}'. Súbanlo a la sesión o corrijan CONFIG['ruta_datos']. "
            "Este notebook no genera datos sintéticos de reemplazo."
        )
    ext = os.path.splitext(ruta_archivo)[1].lower()
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(ruta_archivo)
    if ext in (".csv", ".txt"):
        return pd.read_csv(ruta_archivo)
    if ext in (".parquet",):
        return pd.read_parquet(ruta_archivo)
    raise ValueError(f"Extensión no soportada: '{ext}'")


def preparar_datos(cfg: dict):
    """Carga, limpia, controla calidad y devuelve (df, columnas, informe)."""
    bruto = leer_tabla(cfg["ruta_datos"])
    col = resolver_columnas(bruto, cfg["alias_columnas"])
    informe = {"filas_leidas": int(len(bruto)), "columnas_resueltas": col}
    print(f"Filas leídas        : {len(bruto):,}")
    print(f"Columnas resueltas  : {col}")

    df = bruto.copy()
    for campo in ("titulo", "descripcion"):
        df[col[campo]] = df[col[campo]].fillna("").astype(str).str.strip()

    # --- texto que consume el modelo ---
    piezas = [df[col[c]] for c in cfg["campos_texto"] if c in col]
    texto = piezas[0]
    for pieza in piezas[1:]:
        texto = texto.str.cat(pieza, sep=". ")
    df["texto"] = texto.str.replace(r"\.\s*\.", ".", regex=True).str.strip()

    # --- filas inservibles ---
    n = len(df)
    df = df[df["texto"].str.len() > 0]
    df[col["etiqueta"]] = pd.to_numeric(df[col["etiqueta"]], errors="coerce")
    df = df[df[col["etiqueta"]].isin([0, 1])].copy()
    df[col["etiqueta"]] = df[col["etiqueta"]].astype(int)
    informe["descartadas_texto_o_etiqueta"] = int(n - len(df))
    print(f"Descartadas (texto vacío o etiqueta inválida): {n - len(df):,}")

    # --- fechas ---
    informe["fechas_imposibles"] = 0
    if "fecha" in col:
        df[col["fecha"]] = pd.to_datetime(
            df[col["fecha"]], errors="coerce", dayfirst=cfg["fecha_dia_primero"]
        )
        validas = df[col["fecha"]].notna()
        if validas.any():
            anio_min, anio_max = cfg["anios_validos"]
            fuera = validas & (~df[col["fecha"]].dt.year.between(anio_min, anio_max))
            informe["fechas_imposibles"] = int(fuera.sum())
            print(f"Fechas: {df[col['fecha']].min().date()} a {df[col['fecha']].max().date()}"
                  f"  ({int((~validas).sum()):,} sin parsear)")
            if fuera.any():
                print(f"AVISO: {int(fuera.sum()):,} fechas fuera de [{anio_min}, {anio_max}]. "
                      "La partición temporal quedaría corrompida; revisar el corpus.")
        else:
            print("AVISO: ninguna fecha pudo parsearse.")

    # --- duplicados y contradicciones, sobre texto normalizado ---
    df["texto_norm"] = df["texto"].map(normalizar)
    n = len(df)
    df = df.drop_duplicates(subset=["texto_norm", col["etiqueta"]])
    informe["duplicados_eliminados"] = int(n - len(df))
    print(f"Duplicados eliminados (texto normalizado): {n - len(df):,}")

    conflicto = df.groupby("texto_norm")[col["etiqueta"]].nunique()
    textos_en_conflicto = conflicto[conflicto > 1].index
    n = len(df)
    df = df[~df["texto_norm"].isin(textos_en_conflicto)]
    informe["contradicciones_eliminadas"] = int(n - len(df))
    if n != len(df):
        print(f"AVISO: {n - len(df):,} filas con el mismo texto y etiquetas opuestas. Eliminadas.")

    df = df.reset_index(drop=True)
    df.insert(0, "row_id", np.arange(len(df)))
    informe["n_final"] = int(len(df))
    return df, col, informe


datos, COL, informe_datos = preparar_datos(CONFIG)

if CONFIG["submuestra"]:
    datos = datos.sample(n=min(CONFIG["submuestra"], len(datos)),
                         random_state=SEMILLA_BASE).reset_index(drop=True)
    datos["row_id"] = np.arange(len(datos))
    print(f"\nAVISO: submuestra activa ({len(datos):,} filas). Solo para pruebas.")

COL_ETIQUETA = COL["etiqueta"]
NOMBRES_CLASE = [CONFIG["etiquetas"][0], CONFIG["etiquetas"][1]]

print(f"\nCorpus final: {len(datos):,} noticias")
for k, v in datos[COL_ETIQUETA].value_counts().sort_index().items():
    print(f"  {k} ({CONFIG['etiquetas'][k]:>5}): {v:6,}  ({v / len(datos) * 100:5.2f}%)")

longitudes = datos["texto"].str.split().str.len()
print(f"\nLongitud en palabras: mediana {int(longitudes.median())}, "
      f"p95 {int(longitudes.quantile(0.95))}, máx {int(longitudes.max())}")
print("\nEjemplo de texto que verá el modelo:")
print("  " + datos["texto"].iloc[0][:220] + "...")

---
## §5 · Agrupamiento anti-fuga

### El problema

El artículo fabrica buena parte de sus noticias falsas alterando noticias reales: cambia el nombre
de un político con spaCy, o una cifra con expresiones regulares. El resultado son pares casi
idénticos con etiquetas opuestas:

| Etiqueta | Titular |
|---|---|
| 1 | *Un manifiesto llama a que Xavier Domènech lidere un 'nuevo' Podem Catalunya* |
| 0 | *Un manifiesto llama a que Adriana Lastra lidere un 'nuevo' Podem Catalunya* |

Si un reparto aleatorio manda el primero a entrenamiento y el segundo a prueba, el modelo ya vio
prácticamente ese texto. La métrica sube sin que haya aprendido nada generalizable.

### La solución

Se construye un grafo de variantes y se parte por componentes conexas, no por filas. Cada noticia
se conecta con su titular normalizado y con su descripción normalizada; dos noticias que comparten
cualquiera de los dos quedan en la misma componente. Se resuelve con union-find, que es lineal en
la práctica. Después, la partición mueve **grupos completos**, nunca filas sueltas.

El guardarraíl del final avisa si una sola componente se traga más filas de las que caben en el
split más pequeño: eso ocurre cuando los textos son muy repetitivos y encadenan todo el corpus.

In [ ]:
def asignar_grupos(df: pd.DataFrame, col: dict, agrupar: bool) -> np.ndarray:
    """Union-find sobre titular y descripción normalizados."""
    if not agrupar:
        return np.arange(len(df))

    padre = {}

    def raiz(x):
        padre.setdefault(x, x)
        while padre[x] != x:
            padre[x] = padre[padre[x]]      # compresión de caminos
            x = padre[x]
        return x

    def unir(a, b):
        ra, rb = raiz(a), raiz(b)
        if ra != rb:
            padre[ra] = rb

    titulos = df[col["titulo"]].map(normalizar).values
    descripciones = df[col["descripcion"]].map(normalizar).values

    for i in range(len(df)):
        if titulos[i]:
            unir(("fila", i), ("tit", titulos[i]))
        if descripciones[i]:
            unir(("fila", i), ("des", descripciones[i]))

    raices = [raiz(("fila", i)) for i in range(len(df))]
    codigo = {r: k for k, r in enumerate(dict.fromkeys(raices))}
    return np.array([codigo[r] for r in raices])


datos["grupo"] = asignar_grupos(datos, COL, CONFIG["agrupar_variantes"])

tam_grupos = datos["grupo"].value_counts()
filas_agrupadas = int(tam_grupos[tam_grupos > 1].sum())
pct_mayor = tam_grupos.max() / len(datos) * 100

print(f"Grupos formados       : {datos['grupo'].nunique():,}")
print(f"Grupos con >1 noticia : {int((tam_grupos > 1).sum()):,}  ({filas_agrupadas:,} filas)")
print(f"Grupo más grande      : {int(tam_grupos.max()):,} filas ({pct_mayor:.2f}% del corpus)")
print(f"\n{filas_agrupadas:,} filas ({filas_agrupadas / len(datos) * 100:.1f}%) habrían podido "
      "generar fuga con un reparto aleatorio ingenuo.")

---
## §6 · Partición train / validación / prueba

### Proporciones

Si `CONFIG["proporciones"]` es `None`, se eligen según el tamaño del corpus:

- **80 / 10 / 10** para corpus grandes, que es la traducción directa del 80/20 del artículo
  reservando además un conjunto de validación explícito.
- **70 / 15 / 15** por debajo de `umbral_corpus_pequeno`. Con pocos cientos de noticias, un 10% de
  prueba deja unas decenas de ejemplos y la métrica pasa a tener un margen de error de más de diez
  puntos: cualquier comparación entre modelos se vuelve indistinguible del ruido.

### Estrategias

- **`"grupo"`** — `StratifiedGroupKFold` de scikit-learn, que mantiene a la vez la proporción de
  clases en los tres splits y la integridad de los grupos de la sección anterior. Se aplica dos
  veces: primero se separa prueba, después validación del resto.
- **`"temporal"`** — ordena por fecha y corta: lo antiguo a entrenamiento, lo reciente a prueba.
  Responde a una pregunta distinta y más exigente —si el modelo sirve para detectar desinformación
  futura— y es la más honesta cuando el corpus tiene fechas fiables. Puede dejar fuga residual si un
  par real/falsa quedó a ambos lados del corte; el diagnóstico lo cuantifica en vez de ocultarlo.

El conjunto de prueba no se vuelve a tocar hasta la evaluación final.

In [ ]:
def proporciones_efectivas(cfg: dict, n: int):
    if cfg["proporciones"] is not None:
        return list(cfg["proporciones"]), "declaradas en CONFIG"
    if n < cfg["umbral_corpus_pequeno"]:
        return [0.70, 0.15, 0.15], f"automáticas (corpus pequeño: {n:,} < {cfg['umbral_corpus_pequeno']:,})"
    return [0.80, 0.10, 0.10], f"automáticas (corpus grande: {n:,})"


def _separar_fraccion(y, grupos, fraccion, semilla):
    """Extrae una fracción del conjunto respetando estratificación e integridad de grupos."""
    n_splits = max(2, int(round(1 / fraccion)))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=semilla)
    resto, seleccion = next(sgkf.split(np.zeros(len(y)), y, groups=grupos))
    return resto, seleccion


def particionar(df: pd.DataFrame, col: dict, cfg: dict, proporciones, semilla: int) -> np.ndarray:
    p_train, p_val, p_test = proporciones
    y = df[col["etiqueta"]].values
    splits = pd.Series("train", index=df.index)

    if cfg["estrategia_particion"] == "grupo":
        g = df["grupo"].values
        resto, test = _separar_fraccion(y, g, p_test, semilla)
        frac_val = p_val / (p_train + p_val)
        _, val_rel = _separar_fraccion(y[resto], g[resto], frac_val, semilla)
        splits.iloc[resto[val_rel]] = "val"
        splits.iloc[test] = "test"

    elif cfg["estrategia_particion"] == "temporal":
        if "fecha" not in col or df[col["fecha"]].isna().all():
            raise ValueError("Partición temporal solicitada pero no hay fechas válidas.")
        orden = df.sort_values(col["fecha"], kind="mergesort").index.values
        c1, c2 = int(len(orden) * p_train), int(len(orden) * (p_train + p_val))
        splits.iloc[orden[c1:c2]] = "val"
        splits.iloc[orden[c2:]] = "test"

    else:
        raise ValueError(f"Estrategia no reconocida: {cfg['estrategia_particion']}")

    return splits.values


PROPORCIONES, motivo_prop = proporciones_efectivas(CONFIG, len(datos))
datos["split"] = particionar(datos, COL, CONFIG, PROPORCIONES, SEMILLA_BASE)

print(f"Estrategia   : {CONFIG['estrategia_particion']}")
print(f"Proporciones : {PROPORCIONES}  ({motivo_prop})")

### Diagnóstico de la partición

Tres comprobaciones que van al informe tal cual:

1. **Proporciones y estratificación**: que salió lo que se pidió, con margen de cinco puntos.
2. **Fuga**: ningún grupo repartido entre dos splits.
3. **Piso trivial**: la exactitud que obtiene un clasificador que siempre responde la clase
   mayoritaria. Es la referencia mínima absoluta. En el corpus del artículo está en 58%; sobre los
   63 titulares independientes con los que ellos reportan 71%, el piso era 52,4%, de modo que la
   ganancia real sobre no hacer nada es de unos dieciocho puntos, no de setenta y uno.

In [ ]:
def diagnosticar_particion(df: pd.DataFrame, col: dict, cfg: dict, proporciones):
    tabla = pd.crosstab(df["split"], df[col["etiqueta"]])
    tabla.columns = [f"{c}_{cfg['etiquetas'][c]}" for c in tabla.columns]
    tabla["total"] = tabla.sum(axis=1)
    tabla["%_corpus"] = (tabla["total"] / len(df) * 100).round(2)
    if f"1_{cfg['etiquetas'][1]}" in tabla.columns:
        tabla["%_real"] = (tabla[f"1_{cfg['etiquetas'][1]}"] / tabla["total"] * 100).round(2)
    tabla = tabla.loc[[s for s in ("train", "val", "test") if s in tabla.index]]

    grupos_partidos = int((df.groupby("grupo")["split"].nunique() > 1).sum())

    desvios = []
    for split, pedida in zip(("train", "val", "test"), proporciones):
        obtenida = (df["split"] == split).mean()
        if abs(obtenida - pedida) > 0.05:
            desvios.append(f"{split}: pedido {pedida:.0%}, obtenido {obtenida:.0%}")

    return tabla, grupos_partidos, desvios


tabla_particion, grupos_partidos, desvios = diagnosticar_particion(datos, COL, CONFIG, PROPORCIONES)
print(tabla_particion.to_string())

print(f"\nGrupos repartidos entre splits: {grupos_partidos}")
print("  OK: sin fuga entre particiones." if grupos_partidos == 0 else
      "  AVISO: hay grupos partidos. Esperable en partición temporal; documentarlo.")

if desvios:
    print("\nAVISO: las proporciones se desviaron más de cinco puntos:")
    for d in desvios:
        print(f"    {d}")
    print("    Causa habitual: un grupo demasiado grande (ver §5).")
else:
    print("  OK: proporciones dentro de lo pedido (±5 puntos).")

if "fecha" in COL and datos[COL["fecha"]].notna().any():
    print("\nRango temporal por split:")
    print(datos.groupby("split")[COL["fecha"]].agg(["min", "max"])
          .loc[[s for s in ("train", "val", "test")]].to_string())

# --- piso trivial sobre prueba ---
y_test_prev = datos.loc[datos["split"] == "test", COL_ETIQUETA]
proporcion_test = y_test_prev.value_counts(normalize=True)
PISO_TRIVIAL = float(proporcion_test.max())
CLASE_MAYORITARIA = int(proporcion_test.idxmax())

print()
seccion("PISO TRIVIAL SOBRE EL CONJUNTO DE PRUEBA")
for k, v in proporcion_test.sort_index().items():
    print(f"  Responder siempre '{CONFIG['etiquetas'][k]}': accuracy = {v * 100:5.2f}%")
print(f"\n  Cualquier modelo debe superar {PISO_TRIVIAL * 100:.2f}% con claridad para justificarse.")

datos[["row_id", "split", "grupo", COL_ETIQUETA]].to_csv(ruta("particion.csv"), index=False)
tabla_particion.to_csv(ruta("resumen_particion.csv"))
print(f"\nGuardados 'particion.csv' y 'resumen_particion.csv' en ./{DIR_SALIDA}/")

---
## §7 · Métricas comunes

Los dos modelos se miden exactamente con la misma función, de modo que los números son comparables
sin asteriscos.

### Qué se reporta y por qué

- **Accuracy** — proporción de aciertos. Engañosa con clases desbalanceadas.
- **F1-macro** — media armónica de precisión y recuperación, promediando las dos clases por igual.
  Es la métrica principal aquí, porque el coste del error es asimétrico en ambos sentidos.
- **MCC (Matthews)** — usa las cuatro celdas de la matriz de confusión y vale 0 para un clasificador
  degenerado. Es la métrica que delata el problema cuando accuracy y F1 lo esconden: en la Tabla 5
  del artículo, RoBERTa-multilingual sin ajuste fino tiene recuperación 1 y F1 de 0,745 con MCC de 0,
  simplemente porque respondía "real" a todo.
- **Precisión y recuperación por clase, en las dos convenciones.** El artículo toma como positivo la
  noticia real (etiqueta 1); un sistema de detección toma como positivo la noticia falsa (etiqueta 0).
  Se reportan ambas y se dice cuál es cuál.

### Intervalos de confianza

Con un conjunto de prueba de decenas de ejemplos, una diferencia de tres puntos entre dos modelos
puede ser puro azar. `ic_bootstrap` remuestrea el conjunto de prueba con reemplazo y devuelve el
intervalo al 95%. **Ese intervalo debe aparecer en el informe junto a cada cifra.**

### McNemar

Compara dos modelos sobre los mismos ejemplos. Solo mira los casos en que discrepan: cuántos acierta
uno y falla el otro, y viceversa. Si el reparto entre esas dos casillas es compatible con lanzar una
moneda, la diferencia entre los modelos no es significativa por más que las exactitudes difieran.

In [ ]:
def calcular_metricas(y_true, y_pred, cfg: dict) -> dict:
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }
    for k, nombre in cfg["etiquetas"].items():
        m[f"precision_{nombre}"] = precision_score(y_true, y_pred, pos_label=k, zero_division=0)
        m[f"recall_{nombre}"] = recall_score(y_true, y_pred, pos_label=k, zero_division=0)
        m[f"f1_{nombre}"] = f1_score(y_true, y_pred, pos_label=k, zero_division=0)
    return {k: float(round(v, 4)) for k, v in m.items()}


def es_degenerado(y_pred) -> bool:
    """True si el modelo predice una sola clase para todo el conjunto."""
    return len(np.unique(y_pred)) < 2


def ic_bootstrap(y_true, y_pred, metrica="f1_macro", n=2000, semilla=16, alfa=0.05):
    """Intervalo de confianza por remuestreo con reemplazo del conjunto de evaluación."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rng = np.random.default_rng(semilla)
    valores = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        yt, yp = y_true[idx], y_pred[idx]
        if metrica == "accuracy":
            valores[i] = accuracy_score(yt, yp)
        elif metrica == "mcc":
            valores[i] = matthews_corrcoef(yt, yp) if len(np.unique(yt)) > 1 else 0.0
        else:
            valores[i] = f1_score(yt, yp, average="macro", zero_division=0)
    return float(np.quantile(valores, alfa / 2)), float(np.quantile(valores, 1 - alfa / 2))


def mcnemar(y_true, pred_a, pred_b):
    """Prueba exacta de McNemar. Devuelve (solo_acierta_a, solo_acierta_b, p_valor)."""
    ok_a = np.asarray(pred_a) == np.asarray(y_true)
    ok_b = np.asarray(pred_b) == np.asarray(y_true)
    b = int((ok_a & ~ok_b).sum())
    c = int((~ok_a & ok_b).sum())
    if b + c == 0:
        return b, c, 1.0
    try:
        from scipy.stats import binomtest
        p = binomtest(min(b, c), b + c, 0.5).pvalue
    except Exception:
        from math import comb
        k, n = min(b, c), b + c
        p = min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)
    return b, c, float(p)


def linea_metricas(nombre: str, m: dict, ic=None) -> str:
    texto = (f"{nombre:<34} acc {m['accuracy']:.4f} · F1m {m['f1_macro']:.4f} · "
             f"MCC {m['mcc']:.4f}")
    if ic is not None:
        texto += f"  [IC95% F1m: {ic[0]:.4f}-{ic[1]:.4f}]"
    return texto


print("Funciones de métricas listas (incluyen bootstrap y McNemar).")

---
## §8 · Línea base clásica: TF-IDF y clasificadores lineales

### Por qué va primero

Sin esto, la cifra del modelo profundo no se puede interpretar: lo único con lo que se podría
comparar es el piso trivial, y superarlo no demuestra nada. La pregunta que importa es cuánto de ese
resultado lo consigue ya una bolsa de palabras que entrena en segundos y no necesita GPU.

Como además corre en un abrir y cerrar de ojos, tenerla resuelta antes de encender la GPU permite
detectar de inmediato si el problema está en los datos o en el modelo profundo. En la corrida
anterior, el clásico llegaba a F1-macro 0,887 mientras el RoBERTa se quedaba en 0,339: esa
comparación era el diagnóstico, no un adorno.

### El error que hay que evitar

`TfidfVectorizer` se ajusta **solo sobre entrenamiento**. Ajustarlo sobre todo el corpus es una fuga
sutil: el vocabulario y las estadísticas de frecuencia inversa incorporarían información de los
documentos de prueba. No rompe nada ni lanza error, simplemente infla la métrica.

### Qué se explora

`ngram_range` (solo palabras o palabras más bigramas), `min_df` (umbral de frecuencia mínima) y `C`
(fuerza de la regularización). La rejilla se recorre midiendo en **validación**; el conjunto de
prueba se toca una sola vez, al final.

In [ ]:
def vectorizar(ngramas, min_df, cfg, textos_train, *otros):
    """Ajusta el vectorizador SOLO sobre train y transforma el resto."""
    vec = TfidfVectorizer(
        ngram_range=ngramas, min_df=min_df,
        max_features=cfg["clasico_max_features"],
        sublinear_tf=True, strip_accents="unicode", lowercase=True,
    )
    X_train = vec.fit_transform(textos_train)
    return vec, X_train, [vec.transform(t) for t in otros]


def construir_clasificador(nombre, C, semilla):
    if nombre == "logreg":
        return LogisticRegression(C=C, max_iter=2000, solver="liblinear", random_state=semilla)
    if nombre == "svm":
        return LinearSVC(C=C, max_iter=5000, random_state=semilla)
    if nombre == "nb":
        return MultinomialNB(alpha=1.0)          # no usa C
    raise ValueError(f"Clasificador desconocido: {nombre}")


CLASICOS = {
    "logreg": "TF-IDF + Regresión logística",
    "svm": "TF-IDF + SVM lineal",
    "nb": "TF-IDF + Naive Bayes",
}

partes = {s: datos[datos["split"] == s] for s in ("train", "val", "test")}
TXT = {s: p["texto"].values for s, p in partes.items()}
Y = {s: p[COL_ETIQUETA].values for s, p in partes.items()}
ID = {s: p["row_id"].values for s, p in partes.items()}

t0 = time.time()
filas_busqueda = []
for ngramas in CONFIG["clasico_ngramas"]:
    for min_df in CONFIG["clasico_min_df"]:
        vec, X_tr, (X_va,) = vectorizar(ngramas, min_df, CONFIG, TXT["train"], TXT["val"])
        for nombre in CLASICOS:
            for C in (CONFIG["clasico_C"] if nombre != "nb" else [None]):
                fijar_semillas(SEMILLA_BASE)
                clf = construir_clasificador(nombre, C, SEMILLA_BASE).fit(X_tr, Y["train"])
                m = calcular_metricas(Y["val"], clf.predict(X_va), CONFIG)
                filas_busqueda.append({"modelo": nombre, "ngramas": str(ngramas), "min_df": min_df,
                                       "C": C, "n_vocab": len(vec.vocabulary_),
                                       **{f"val_{k}": v for k, v in m.items()}})

busqueda = pd.DataFrame(filas_busqueda)
criterio = f"val_{CONFIG['metrica_seleccion']}"
mejores_clasicos = {n: busqueda[busqueda["modelo"] == n].sort_values(criterio, ascending=False)
                    .iloc[0].to_dict() for n in CLASICOS}

print(f"{len(busqueda)} configuraciones exploradas en {time.time() - t0:.1f} s.\n")
print(f"Mejor configuración de cada clasificador según {CONFIG['metrica_seleccion']} en VALIDACIÓN:\n")
print(pd.DataFrame(list(mejores_clasicos.values()))[
    ["modelo", "ngramas", "min_df", "C", "n_vocab", criterio, "val_accuracy", "val_mcc"]
].to_string(index=False))

busqueda.to_csv(ruta("busqueda_clasicos.csv"), index=False)

### Evaluación sobre prueba

Cada clasificador se reentrena con su mejor configuración y se evalúa una sola vez. Se entrena solo
con `train`, no con `train + val`: juntarlos daría un modelo algo mejor, pero rompería la igualdad
de condiciones con el modelo profundo, que también entrena solo con `train`.

In [ ]:
metricas_clasicos, predicciones_clasicos = {}, {}
modelos_clasicos, vectorizadores_clasicos = {}, {}

for nombre, cfg_m in mejores_clasicos.items():
    ngramas, min_df, C = eval(cfg_m["ngramas"]), int(cfg_m["min_df"]), cfg_m["C"]
    vec, X_tr, (X_te,) = vectorizar(ngramas, min_df, CONFIG, TXT["train"], TXT["test"])
    fijar_semillas(SEMILLA_BASE)
    clf = construir_clasificador(nombre, C, SEMILLA_BASE).fit(X_tr, Y["train"])

    pred = clf.predict(X_te)
    puntaje = (clf.predict_proba(X_te)[:, 1] if hasattr(clf, "predict_proba")
               else clf.decision_function(X_te))

    metricas_clasicos[nombre] = calcular_metricas(Y["test"], pred, CONFIG)
    predicciones_clasicos[nombre] = {"pred": pred, "puntaje": puntaje}
    modelos_clasicos[nombre] = clf
    vectorizadores_clasicos[nombre] = vec

MEJOR_CLASICO = max(CLASICOS, key=lambda n: metricas_clasicos[n][CONFIG["metrica_seleccion"]])

seccion(f"LÍNEA BASE CLÁSICA — CONJUNTO DE PRUEBA (n = {len(Y['test']):,})")
for nombre in CLASICOS:
    m = metricas_clasicos[nombre]
    ic = ic_bootstrap(Y["test"], predicciones_clasicos[nombre]["pred"],
                      "f1_macro", CONFIG["bootstrap_n"], SEMILLA_BASE)
    marca = "  <- mejor" if nombre == MEJOR_CLASICO else ""
    print(linea_metricas(CLASICOS[nombre], m, ic) + marca)

m_mejor = metricas_clasicos[MEJOR_CLASICO]
print(f"\nPiso trivial: {PISO_TRIVIAL:.4f}   ·   "
      f"mejora del mejor clásico: {(m_mejor['accuracy'] - PISO_TRIVIAL) * 100:+.2f} puntos")

print(f"\nReporte por clase de {CLASICOS[MEJOR_CLASICO]}:\n")
print(classification_report(Y["test"], predicciones_clasicos[MEJOR_CLASICO]["pred"],
                            target_names=NOMBRES_CLASE, digits=4, zero_division=0))

---
## §9 · Términos indicativos: la sonda del atajo

La regresión logística asigna un coeficiente a cada término: positivo empuja hacia `real`, negativo
hacia `falsa`. Ordenar esos coeficientes da literalmente la lista de palabras en las que el modelo
se apoya para decidir, sin necesidad de mapas de atención.

La pregunta que hay que hacerle a la lista: **¿estos términos hablan de qué se dice, o de cómo está
escrito y quién lo escribió?**

| Si dominan… | Interpretación |
|---|---|
| Nombres propios de personas y partidos, repartidos de forma asimétrica | Firma de la generación por sustitución de entidades. El clasificador no juzga veracidad: reconoce qué nombres aparecen. Es el hallazgo más grave posible. |
| Conectores, adverbios, verbos genéricos, muletillas | El modelo detecta registro de redacción. Si parte de las falsas se generó con un modelo de lenguaje, es la huella del generador. |
| Cifras, fechas, siglas y vocabulario institucional del lado "real" | Las verdaderas vienen de raspado web y de fuentes oficiales, y arrastran vocabulario que las falsas no tienen. Mismo problema por el otro lado. |
| Temas concretos, sin ninguno de los tres patrones anteriores | El modelo aprende que ciertos asuntos aparecen más en desinformación. Discutible, pero es señal sustantiva. |

La segunda mitad de la celda cierra el argumento cuantitativamente: un coeficiente alto no basta,
porque un término puede pesar mucho y aun así aparecer en ambas clases. Lo que delata la sustitución
de entidades es la **asimetría de aparición**, que un término salga casi solo en un lado.

In [ ]:
def terminos_indicativos(clf, vec, top=20):
    terminos = np.array(vec.get_feature_names_out())
    coefs = clf.coef_[0]
    idx_real = np.argsort(coefs)[-top:][::-1]
    idx_falsa = np.argsort(coefs)[:top]
    tabla = pd.DataFrame({
        f"empuja a '{CONFIG['etiquetas'][1]}'": terminos[idx_real],
        "coef_real": np.round(coefs[idx_real], 3),
        f"empuja a '{CONFIG['etiquetas'][0]}'": terminos[idx_falsa],
        "coef_falsa": np.round(coefs[idx_falsa], 3),
    })
    return tabla, terminos, coefs, idx_real, idx_falsa


def exclusividad(vec, terminos, indices, textos_train, y_train, top=15):
    """Porcentaje de noticias de cada clase en que aparece cada término."""
    presencia = vec.transform(textos_train) > 0
    mask_falsa, mask_real = (y_train == 0), (y_train == 1)
    filas = []
    for idx, lado in indices:
        for i in idx[:top]:
            col = presencia[:, i]
            p_f = float(col[mask_falsa].sum()) / mask_falsa.sum() * 100
            p_r = float(col[mask_real].sum()) / mask_real.sum() * 100
            razon = (p_f + 1e-6) / (p_r + 1e-6)
            filas.append({"termino": terminos[i], "empuja_a": lado,
                          "%_en_falsas": round(p_f, 2), "%_en_reales": round(p_r, 2),
                          "razon_falsa_real": round(razon, 2),
                          "casi_exclusivo": bool(razon > 10 or razon < 0.1)})
    return pd.DataFrame(filas)


if "logreg" in modelos_clasicos:
    clf_lr, vec_lr = modelos_clasicos["logreg"], vectorizadores_clasicos["logreg"]
    tabla_terminos, terminos, coefs, idx_real, idx_falsa = terminos_indicativos(clf_lr, vec_lr)

    print(f"Términos más indicativos (regresión logística, ngramas "
          f"{mejores_clasicos['logreg']['ngramas']}, vocabulario "
          f"{int(mejores_clasicos['logreg']['n_vocab']):,})\n")
    print(tabla_terminos.to_string(index=False))
    tabla_terminos.to_csv(ruta("terminos_indicativos.csv"), index=False)

    tabla_exc = exclusividad(vec_lr, terminos, [(idx_falsa, "falsa"), (idx_real, "real")],
                             TXT["train"], Y["train"])
    n_exc = int(tabla_exc["casi_exclusivo"].sum())
    print(f"\nAparición por clase sobre entrenamiento "
          f"({int((Y['train'] == 0).sum()):,} falsas · {int((Y['train'] == 1).sum()):,} reales):\n")
    print(tabla_exc.to_string(index=False))
    print(f"\nTérminos casi exclusivos de una clase: {n_exc} de {len(tabla_exc)} "
          f"({n_exc / len(tabla_exc) * 100:.0f}%)")
    if n_exc / len(tabla_exc) > 0.5:
        print("  Más de la mitad de los términos más informativos aparecen casi solo en una clase:")
        print("  el corpus permite separar las clases reconociendo vocabulario, sin evaluar si lo")
        print("  que dice la noticia es cierto. Es el hallazgo central para el informe.")
    else:
        print("  El vocabulario se reparte entre clases: la señal es de frecuencia, no de")
        print("  exclusividad. El atajo es menos evidente por esta vía.")
    tabla_exc.to_csv(ruta("exclusividad_terminos.csv"), index=False)

    peso = np.sort(np.abs(coefs))[::-1]
    print("\nConcentración del peso del modelo:")
    for k in (10, 100, 1000):
        if k <= len(peso):
            print(f"  Los {k:>5,} términos de mayor peso concentran "
                  f"{peso[:k].sum() / peso.sum() * 100:5.1f}% del total")

---
## §10 · Resolución del modelo preentrenado

### El problema

El artículo usa `PlanTL-GOB-ES/roberta-base-bne`. Ese repositorio ya no contiene el modelo: fue
marcado como obsoleto y sus archivos eliminados, de modo que solo quedan `README.md` y
`.gitattributes`. `AutoConfig` falla porque no hay `config.json` que leer, y `force_download=True`
no ayuda: fuerza a redescargar un repositorio vacío.

### La solución

Se prueban varios repositorios en orden, comprobando los archivos remotos antes de descargar nada,
y se usa el primero que tenga a la vez `config.json` y pesos. El orden empieza por el repositorio
canónico, así que si Hugging Face lo restaura el notebook vuelve a usarlo sin tocar nada. Se
registran el repositorio efectivo y el hash del commit.

### Dos verificaciones, no una

1. **Arquitectura** — que el espejo declare lo mismo que la ficha oficial: 12 capas, dimensión
   oculta 768, 12 cabezas de atención y un vocabulario de 50.262 tokens (cifra peculiar, distinta de
   los 50.265 del RoBERTa en inglés, que sirve de huella).
2. **Pesos** — que la arquitectura coincida no prueba que los pesos sean los del modelo entrenado
   por el Barcelona Supercomputing Center: un repositorio con pesos aleatorios pasaría la primera
   comprobación sin problema. Por eso se carga la cabeza de modelado de lenguaje enmascarado y se le
   pide completar frases en español. Si las predicciones son sensatas, los pesos son reales; si son
   ruido, el ajuste fino estaría partiendo de cero y eso explicaría por sí solo un resultado plano.

Que el modelo del artículo haya desaparecido de su repositorio a menos de dos años de la publicación
es un hallazgo reportable sobre reproducibilidad, no un contratiempo. Conviene documentarlo.

In [ ]:
from huggingface_hub import list_repo_files, model_info

ARCHIVOS_PESOS = ("model.safetensors", "pytorch_model.bin",
                  "model.safetensors.index.json", "pytorch_model.bin.index.json")

FICHA_OFICIAL = {"model_type": "roberta", "vocab_size": 50262, "num_hidden_layers": 12,
                 "hidden_size": 768, "num_attention_heads": 12, "max_position_embeddings": 514}


def repo_utilizable(repo_id):
    try:
        archivos = set(list_repo_files(repo_id))
    except Exception as e:
        return False, type(e).__name__, None
    if "config.json" not in archivos:
        return False, "sin config.json (obsoleto)", None
    if not any(a in archivos for a in ARCHIVOS_PESOS):
        return False, "sin archivos de pesos", None
    try:
        sha = model_info(repo_id).sha
    except Exception:
        sha = None
    return True, "ok", sha


def resolver_modelo(cfg):
    print("Comprobando repositorios candidatos:\n")
    for repo in list(cfg["modelo_candidatos"]) + [cfg["modelo_respaldo"]]:
        ok, motivo, sha = repo_utilizable(repo)
        print(f"  {'USABLE' if ok else 'descartado: ' + motivo:<34} {repo}")
        if ok:
            return repo, sha
    raise RuntimeError("Ningún repositorio candidato tiene pesos descargables. "
                       "Añadan otro espejo a CONFIG['modelo_candidatos'].")


MODELO, MODELO_SHA = resolver_modelo(CONFIG)
ES_CANONICO = (MODELO == CONFIG["modelo_candidatos"][0])
print(f"\nModelo resuelto : {MODELO}")
print(f"Commit          : {MODELO_SHA}")
if not ES_CANONICO:
    print(f"AVISO: no se usa el repositorio del artículo "
          f"('{CONFIG['modelo_candidatos'][0]}'), que fue vaciado. Declararlo en el informe.")

# --- verificación 1: arquitectura ---
if CONFIG["verificar_arquitectura"] and not ES_CANONICO:
    cfg_remota = AutoConfig.from_pretrained(MODELO)
    print("\nFicha arquitectónica frente a la oficial de roberta-base-bne:\n")
    discrepancias = []
    for clave, esperado in FICHA_OFICIAL.items():
        obtenido = getattr(cfg_remota, clave, None)
        coincide = obtenido == esperado
        print(f"  {'ok ' if coincide else 'NO '} {clave:<24} esperado {esperado!s:<10} obtenido {obtenido}")
        if not coincide:
            discrepancias.append(clave)
    if not discrepancias:
        print("\n  La ficha coincide en todo: el espejo es consistente con el modelo del artículo.")
    elif MODELO == CONFIG["modelo_respaldo"]:
        print(f"\n  Se está usando el modelo de respaldo ({MODELO}), que es otro de los siete del")
        print("  artículo. Las diferencias de ficha son esperadas; reportar el cambio de modelo.")
    else:
        print(f"\n  AVISO: discrepancias en {discrepancias}. El espejo puede no ser fiel.")

# --- verificación 2: los pesos preentrenados son reales ---
if CONFIG["verificar_pesos_con_mlm"]:
    print("\nComprobando que los pesos preentrenados no sean aleatorios:\n")
    tok_tmp = AutoTokenizer.from_pretrained(MODELO)
    mlm = AutoModelForMaskedLM.from_pretrained(MODELO).eval()
    for frase in ("La capital de Colombia es {}.",
                  "El presidente de la República firmó la {} tributaria."):
        entrada = tok_tmp(frase.format(tok_tmp.mask_token), return_tensors="pt")
        pos = (entrada["input_ids"][0] == tok_tmp.mask_token_id).nonzero()[0, 0]
        with torch.no_grad():
            logits = mlm(**entrada).logits
        mejores = logits[0, pos].topk(5).indices
        print(f"  {frase.format('___'):<52} -> "
              f"{[tok_tmp.decode(t).strip() for t in mejores]}")
    del mlm, tok_tmp
    torch.cuda.empty_cache()
    print("\n  Si las palabras son plausibles en español, los pesos son los del modelo preentrenado.")

---
## §11 · Tokenización y carga por lotes

RoBERTa no trabaja con palabras sino con subpalabras (BPE): *"gobierno"* es un solo token,
*"desfinanciación"* se parte en varios fragmentos. Todo lo que exceda `max_seq_length` se trunca y
el modelo nunca lo ve, así que la celda mide qué porcentaje de noticias se está recortando.

**Padding dinámico.** La versión anterior rellenaba todas las noticias hasta 128 tokens cuando la
mediana real del corpus es de 49: dos tercios de cada lote eran relleno, y el modelo gastaba ahí
tiempo de cómputo y atención. Aquí el relleno se calcula por lote, hasta la noticia más larga de ese
lote. Mismo resultado, bastante menos trabajo.

Conviene notar que el artículo usa 128 tokens para los RoBERTa y 512 para los BERT, es decir, compara
familias de modelos que no leen la misma cantidad de texto. Aquí se mantiene 128 por coherencia con la
fila RoBERTa de su Tabla 3, pero la comparación entre familias del artículo queda invalidada por eso.

In [ ]:
tokenizador = AutoTokenizer.from_pretrained(MODELO)
MAX_LEN = CONFIG["hp_articulo"]["max_seq_length"]

muestra = datos["texto"].sample(min(2000, len(datos)), random_state=SEMILLA_BASE).tolist()
largos = np.array([len(tokenizador.encode(t, add_special_tokens=True)) for t in muestra])
PCT_TRUNCADO = float((largos > MAX_LEN).mean() * 100)

print(f"Tokenizador: {tokenizador.__class__.__name__} · vocabulario {tokenizador.vocab_size:,}")
print(f"Longitud en tokens: mediana {int(np.median(largos))}, "
      f"p95 {int(np.percentile(largos, 95))}, máx {int(largos.max())}")
print(f"Noticias truncadas con max_seq_length={MAX_LEN}: {PCT_TRUNCADO:.2f}%")
if PCT_TRUNCADO > 10:
    print("AVISO: truncamiento alto. Considerar subir max_seq_length y documentarlo.")


class ColeccionNoticias(Dataset):
    """Guarda texto crudo; la tokenización ocurre por lote (padding dinámico)."""

    def __init__(self, textos, etiquetas):
        self.textos = list(textos)
        self.etiquetas = list(etiquetas)

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, i):
        return self.textos[i], int(self.etiquetas[i])


def hacer_collate(tok, max_len):
    def collate(lote):
        textos, etiquetas = zip(*lote)
        enc = tok(list(textos), truncation=True, padding=True,
                  max_length=max_len, return_tensors="pt")
        enc["labels"] = torch.tensor(etiquetas, dtype=torch.long)
        return enc
    return collate


COLECCIONES = {s: ColeccionNoticias(TXT[s], Y[s]) for s in ("train", "val", "test")}
collate = hacer_collate(tokenizador, MAX_LEN)


def cargadores(batch_size, semilla):
    """DataLoaders con barajado reproducible. Se rehacen en cada semilla."""
    g = torch.Generator()
    g.manual_seed(semilla)
    return {
        "train": DataLoader(COLECCIONES["train"], batch_size=batch_size, shuffle=True,
                            generator=g, collate_fn=collate),
        "val": DataLoader(COLECCIONES["val"], batch_size=batch_size * 2,
                          shuffle=False, collate_fn=collate),
        "test": DataLoader(COLECCIONES["test"], batch_size=batch_size * 2,
                           shuffle=False, collate_fn=collate),
    }


print(f"\nColecciones listas: train {len(COLECCIONES['train']):,} · "
      f"val {len(COLECCIONES['val']):,} · test {len(COLECCIONES['test']):,}")

---
## §12 · Ajuste fino de RoBERTa-BNE

### Hiperparámetros efectivos

Se parte de la Tabla 3 del artículo y se escala si el corpus es pequeño. La celda imprime los
valores finales y el motivo del cambio, de modo que el informe pueda declarar exactamente qué se
usó y por qué se apartó del artículo.

### Componentes del entrenamiento

- **AdamW**, con el decaimiento de pesos desacoplado del gradiente adaptativo. No se aplica a los
  sesgos ni a las capas de normalización, que es la práctica estándar en el ajuste fino de esta
  familia de modelos.
- **Planificador lineal con calentamiento**: la tasa de aprendizaje sube desde cero durante el primer
  tramo de los pasos y luego baja hasta cero. Evita que los primeros lotes, con la cabeza aún
  aleatoria, destruyan los pesos preentrenados. El artículo no reporta este detalle.
- **Recorte de gradiente** a norma 1,0 y **precisión mixta** cuando hay GPU.

### Selección del mejor punto de control

Al final de cada época se evalúa sobre validación —nunca sobre prueba— y se guarda el estado con
mejor F1-macro, **desempatando por MCC**. El desempate importa: un modelo que predice una sola clase
puede empatar en F1-macro con uno que aprendió algo, y el criterio anterior, que solo miraba F1 con
comparación estricta, se quedaba con el primero que apareciera, que era justamente el degenerado.

Además la parada temprana no puede dispararse antes de `epocas_minimas`. En la corrida anterior el
entrenamiento se cortó en la época 4 con el punto de control de la época 1 —el peor posible— porque
tres épocas planas seguidas cuentan como "sin mejora" aunque el modelo aún no hubiera salido del
colapso inicial.

### Guardarraíl de colapso

Si al terminar el modelo predice una sola clase en validación, se marca la corrida como colapsada.
Si todas las semillas colapsan y `reintentar_si_colapsa` está activo, se reentrena una vez con el
triple de tasa de aprendizaje. Un colapso persistente no es un fallo del código: significa que con
esos datos y ese presupuesto de pasos el modelo no consigue separar nada, y **eso es un resultado
reportable**, sobre todo cuando la línea base clásica sí lo consigue.

In [ ]:
def hiperparametros_efectivos(cfg, n_train):
    hp = dict(cfg["hp_articulo"])
    hp["origen"] = "Tabla 3 del artículo (columna RoBERTa)"
    if cfg["escalar_si_corpus_pequeno"] and n_train < cfg["umbral_corpus_pequeno"]:
        hp.update(cfg["hp_corpus_pequeno"])
        hp["origen"] = (f"Tabla 3 escalada: n_train = {n_train:,} < "
                        f"{cfg['umbral_corpus_pequeno']:,}")
    return hp


def crear_optimizador(modelo, hp, pasos_totales, cfg):
    sin_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    grupos = [
        {"params": [p for n, p in modelo.named_parameters() if not any(s in n for s in sin_decay)],
         "weight_decay": hp["weight_decay"]},
        {"params": [p for n, p in modelo.named_parameters() if any(s in n for s in sin_decay)],
         "weight_decay": 0.0},
    ]
    optim = torch.optim.AdamW(grupos, lr=hp["learning_rate"], eps=cfg["adam_epsilon"])
    sched = get_linear_schedule_with_warmup(
        optim, int(pasos_totales * hp["warmup_ratio"]), pasos_totales)
    return optim, sched


def construir_modelo(hp, cfg):
    config = AutoConfig.from_pretrained(
        MODELO, num_labels=2,
        hidden_dropout_prob=hp["dropout"], attention_probs_dropout_prob=hp["dropout"],
        classifier_dropout=hp["dropout"],
        id2label={int(k): v for k, v in cfg["etiquetas"].items()},
        label2id={v: int(k) for k, v in cfg["etiquetas"].items()},
    )
    return AutoModelForSequenceClassification.from_pretrained(MODELO, config=config).to(DISPOSITIVO)


def escalador_amp(activo):
    try:
        return torch.amp.GradScaler(DISPOSITIVO.type, enabled=activo)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=activo)


@torch.no_grad()
def evaluar(modelo, cargador, cfg):
    """Devuelve (metricas, y_true, y_pred, probabilidades, perdida_media)."""
    modelo.eval()
    preds, reales, probas, perdidas, pesos = [], [], [], [], []
    for lote in cargador:
        lote = {k: v.to(DISPOSITIVO) for k, v in lote.items()}
        with torch.amp.autocast(device_type=DISPOSITIVO.type, enabled=cfg["usar_amp"]):
            salida = modelo(**lote)
        n = lote["labels"].size(0)
        perdidas.append(salida.loss.item() * n)
        pesos.append(n)
        p = torch.softmax(salida.logits.float(), dim=-1)
        probas.append(p.cpu().numpy())
        preds.append(p.argmax(dim=-1).cpu().numpy())
        reales.append(lote["labels"].cpu().numpy())

    y_pred, y_true = np.concatenate(preds), np.concatenate(reales)
    perdida = float(np.sum(perdidas) / np.sum(pesos))
    return calcular_metricas(y_true, y_pred, cfg), y_true, y_pred, np.concatenate(probas), perdida


def entrenar_una_semilla(semilla, hp, cfg, verboso=True):
    """Entrena un modelo completo. Devuelve (modelo, historial, informacion)."""
    fijar_semillas(semilla)
    dl = cargadores(hp["batch_size"], semilla)
    modelo = construir_modelo(hp, cfg)
    pasos = len(dl["train"]) * hp["epocas"]
    optim, sched = crear_optimizador(modelo, hp, pasos, cfg)
    escalador = escalador_amp(cfg["usar_amp"])

    pesos_clase = None
    if cfg["usar_pesos_clase"]:
        conteo = np.bincount(np.array(COLECCIONES["train"].etiquetas), minlength=2)
        pesos_clase = torch.tensor(len(COLECCIONES["train"]) / (2 * conteo),
                                   dtype=torch.float, device=DISPOSITIVO)

    historial, mejor_clave, mejor_epoca, sin_mejora, mejor_estado = [], (-1.0, -1.0), -1, 0, None
    t0 = time.time()

    for epoca in range(1, hp["epocas"] + 1):
        modelo.train()
        perdidas = []
        for lote in dl["train"]:
            lote = {k: v.to(DISPOSITIVO) for k, v in lote.items()}
            optim.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DISPOSITIVO.type, enabled=cfg["usar_amp"]):
                if pesos_clase is None:
                    perdida = modelo(**lote).loss
                else:
                    logits = modelo(**{k: v for k, v in lote.items() if k != "labels"}).logits
                    perdida = torch.nn.functional.cross_entropy(
                        logits, lote["labels"], weight=pesos_clase)
            escalador.scale(perdida).backward()
            escalador.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), cfg["max_grad_norm"])
            escalador.step(optim)
            escalador.update()
            sched.step()
            perdidas.append(perdida.item())

        perdida_train = float(np.mean(perdidas))
        m_val, _, pred_val, _, perdida_val = evaluar(modelo, dl["val"], cfg)
        degenerado = es_degenerado(pred_val)

        historial.append({"epoca": epoca, "train_loss": round(perdida_train, 4),
                          "val_loss": round(perdida_val, 4), "lr": sched.get_last_lr()[0],
                          "degenerado": degenerado,
                          **{f"val_{k}": v for k, v in m_val.items()}})

        clave = (m_val[cfg["metrica_seleccion"]], m_val["mcc"])   # desempate por MCC
        marca = ""
        if clave > mejor_clave:
            mejor_clave, mejor_epoca, sin_mejora = clave, epoca, 0
            mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
            marca = "  <- mejor"
        else:
            sin_mejora += 1

        if verboso:
            print(f"  época {epoca:>2}/{hp['epocas']} · train {perdida_train:.4f} · "
                  f"val {perdida_val:.4f} · acc {m_val['accuracy']:.4f} · "
                  f"F1m {m_val['f1_macro']:.4f} · MCC {m_val['mcc']:.4f}"
                  + ("  [una sola clase]" if degenerado else "") + marca)

        if epoca >= hp["epocas_minimas"] and hp["paciencia"] and sin_mejora >= hp["paciencia"]:
            if verboso:
                print(f"  Parada temprana: {hp['paciencia']} épocas sin mejorar.")
            break

    if mejor_estado is not None:
        modelo.load_state_dict(mejor_estado)

    m_val, _, pred_val, _, _ = evaluar(modelo, dl["val"], cfg)
    info = {"semilla": semilla, "mejor_epoca": mejor_epoca, "epocas_ejecutadas": len(historial),
            "clave_val": mejor_clave, "val": m_val, "colapso": es_degenerado(pred_val),
            "minutos": round((time.time() - t0) / 60, 2)}
    return modelo, pd.DataFrame(historial), info


print("Funciones de entrenamiento listas.")

In [ ]:
HP = hiperparametros_efectivos(CONFIG, int((datos["split"] == "train").sum()))

seccion("HIPERPARÁMETROS EFECTIVOS")
print(f"  Origen              : {HP['origen']}")
for k in ("learning_rate", "epocas", "epocas_minimas", "paciencia", "batch_size",
          "max_seq_length", "dropout", "weight_decay", "warmup_ratio"):
    print(f"  {k:<20}: {HP[k]}")
pasos_epoca = int(np.ceil(len(COLECCIONES["train"]) / HP["batch_size"]))
print(f"  pasos por época     : {pasos_epoca:,}   (total previsto: {pasos_epoca * HP['epocas']:,})")
print(f"  semillas            : {CONFIG['semillas']}")


def entrenar_todas_las_semillas(hp, cfg):
    corridas, mejor_global, mejor_estado = [], None, None
    for semilla in cfg["semillas"]:
        print(f"\n--- semilla {semilla} ---")
        modelo, historial, info = entrenar_una_semilla(semilla, hp, cfg)
        m_test, y_true, y_pred, probas, perdida_test = evaluar(
            modelo, cargadores(hp["batch_size"], semilla)["test"], cfg)
        info["test"] = m_test
        info["historial"] = historial
        corridas.append(info)
        print(f"  fin: mejor época {info['mejor_epoca']} · "
              f"val F1m {info['val']['f1_macro']:.4f} · test F1m {m_test['f1_macro']:.4f} · "
              f"{info['minutos']} min" + ("  [COLAPSO]" if info["colapso"] else ""))

        if mejor_global is None or info["clave_val"] > mejor_global["clave_val"]:
            mejor_global = info
            mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
            mejor_global["test_detalle"] = (y_true, y_pred, probas, perdida_test)
        del modelo
        torch.cuda.empty_cache()
    return corridas, mejor_global, mejor_estado


corridas, mejor_corrida, mejor_estado = entrenar_todas_las_semillas(HP, CONFIG)

if all(c["colapso"] for c in corridas) and CONFIG["reintentar_si_colapsa"]:
    print("\nAVISO: todas las semillas colapsaron (una sola clase predicha).")
    print("Reintentando una vez con el triple de tasa de aprendizaje.\n")
    HP = {**HP, "learning_rate": HP["learning_rate"] * 3,
          "origen": HP["origen"] + " + reintento con lr x3 tras colapso"}
    corridas, mejor_corrida, mejor_estado = entrenar_todas_las_semillas(HP, CONFIG)

historial_mejor = mejor_corrida["historial"]
historial_mejor.to_csv(ruta("historial_mejor_semilla.csv"), index=False)

resumen_semillas = pd.DataFrame([{
    "semilla": c["semilla"], "mejor_epoca": c["mejor_epoca"], "epocas": c["epocas_ejecutadas"],
    "val_f1_macro": c["val"]["f1_macro"], "test_accuracy": c["test"]["accuracy"],
    "test_f1_macro": c["test"]["f1_macro"], "test_mcc": c["test"]["mcc"],
    "colapso": c["colapso"], "minutos": c["minutos"],
} for c in corridas])

print()
seccion("RESULTADO POR SEMILLA")
print(resumen_semillas.to_string(index=False))
for metrica in ("test_accuracy", "test_f1_macro", "test_mcc"):
    print(f"\n{metrica:<16}: media {resumen_semillas[metrica].mean():.4f} "
          f"± {resumen_semillas[metrica].std(ddof=1) if len(resumen_semillas) > 1 else 0:.4f} "
          f"(desviación estándar entre semillas)")
resumen_semillas.to_csv(ruta("resumen_semillas.csv"), index=False)

### Curvas de entrenamiento

Qué hay que leer en ellas para el informe:

- **Convergencia**: si la pérdida de entrenamiento baja y se estabiliza. Si se queda plana en torno
  a 0,69 —que es el logaritmo natural de 2, la pérdida de un clasificador binario que responde al
  azar— el modelo no está aprendiendo nada. Ese fue exactamente el caso en la corrida anterior.
- **Sobreajuste**: si la pérdida de entrenamiento sigue bajando mientras la de validación sube.
- **Estabilidad**: los saltos bruscos sugieren tasa de aprendizaje demasiado alta.
- **Época del mejor punto de control**: si fue la primera o la segunda, conviene sospechar. En este
  problema aprender la tarea de inmediato apunta a un atajo, no a comprensión.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
h = historial_mejor

ejes[0].plot(h["epoca"], h["train_loss"], marker="o", label="Entrenamiento")
ejes[0].plot(h["epoca"], h["val_loss"], marker="s", label="Validación")
ejes[0].axhline(np.log(2), ls=":", c="crimson", alpha=.8)
ejes[0].annotate("ln(2): azar", xy=(h["epoca"].iloc[0], np.log(2)), xytext=(2, 5),
                 textcoords="offset points", fontsize=8, color="crimson")
ejes[0].set_xlabel("Época"); ejes[0].set_ylabel("Pérdida (entropía cruzada)")
ejes[0].set_title(f"Pérdida por época (semilla {mejor_corrida['semilla']})")
ejes[0].legend(); ejes[0].grid(alpha=.3)

ejes[1].plot(h["epoca"], h["val_accuracy"], marker="o", label="Accuracy")
ejes[1].plot(h["epoca"], h["val_f1_macro"], marker="s", label="F1-macro")
ejes[1].plot(h["epoca"], h["val_mcc"], marker="^", label="MCC")
ejes[1].axhline(PISO_TRIVIAL, ls="--", c="gray", alpha=.7)
ejes[1].annotate("piso trivial", xy=(h["epoca"].iloc[0], PISO_TRIVIAL), xytext=(2, 5),
                 textcoords="offset points", fontsize=8, color="gray")
ejes[1].axvline(mejor_corrida["mejor_epoca"], ls="--", c="seagreen", alpha=.6)
ejes[1].set_xlabel("Época"); ejes[1].set_ylabel("Métrica")
ejes[1].set_title("Métricas de validación por época")
ejes[1].legend(); ejes[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig(ruta("curvas_entrenamiento.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## §13 · Evaluación final y comparación entre modelos

El conjunto de prueba no participó en el entrenamiento, ni en la elección del punto de control, ni
en el ajuste de hiperparámetros de ninguno de los dos modelos. Se toca una sola vez, aquí.

Tres lecturas, en orden de importancia:

1. **Cada modelo frente al piso trivial**, con su intervalo de confianza. Si el intervalo del modelo
   incluye el piso, no hay evidencia de que el modelo sirva.
2. **El modelo profundo frente a la línea base clásica.** Si la diferencia es pequeña, hay que
   decirlo con todas las letras: 124 millones de parámetros y una GPU frente a una bolsa de palabras
   que entrena en segundos.
3. **Dónde se equivocan.** El cruce ejemplo por ejemplo distingue cuatro casos: los que resuelven
   ambos (fáciles, resolubles con bolsa de palabras), los que solo resuelve el profundo (el valor
   real de la arquitectura), los que solo resuelve el clásico y los que fallan ambos. Estos últimos
   son el mejor material para el análisis de fallas: si al leerlos resulta que una persona informada
   tampoco podría decidir con solo el texto, el problema no está en el modelo sino en que la tarea,
   planteada así, no es decidible sin consultar fuentes externas.

In [ ]:
y_true_test, y_pred_profundo, probas_profundo, perdida_test = mejor_corrida["test_detalle"]
m_profundo = mejor_corrida["test"]
m_clasico = metricas_clasicos[MEJOR_CLASICO]
pred_clasico = predicciones_clasicos[MEJOR_CLASICO]["pred"]

ic_prof = ic_bootstrap(y_true_test, y_pred_profundo, "f1_macro", CONFIG["bootstrap_n"], SEMILLA_BASE)
ic_clas = ic_bootstrap(Y["test"], pred_clasico, "f1_macro", CONFIG["bootstrap_n"], SEMILLA_BASE)

seccion(f"CONJUNTO DE PRUEBA (n = {len(y_true_test):,})")
print(f"Piso trivial: {PISO_TRIVIAL:.4f}\n")
print(linea_metricas(CLASICOS[MEJOR_CLASICO], m_clasico, ic_clas))
print(linea_metricas(f"RoBERTa-BNE (semilla {mejor_corrida['semilla']})", m_profundo, ic_prof))

comparativa = pd.DataFrame({
    "metrica": ["accuracy", "f1_macro", "mcc", "recall_falsa", "precision_falsa", "recall_real"],
})
comparativa["clasico"] = [m_clasico[k] for k in comparativa["metrica"]]
comparativa["profundo"] = [m_profundo[k] for k in comparativa["metrica"]]
comparativa["diferencia"] = (comparativa["profundo"] - comparativa["clasico"]).round(4)
print("\n" + comparativa.to_string(index=False))

b, c, p_valor = mcnemar(y_true_test, pred_clasico, y_pred_profundo)
print(f"\nMcNemar: {b} casos que solo acierta el clásico, {c} que solo acierta el profundo, "
      f"p = {p_valor:.4f}")
print("  " + ("La diferencia entre ambos modelos es estadísticamente significativa (p < 0,05)."
               if p_valor < 0.05 else
               "No hay evidencia de que un modelo sea mejor que el otro sobre este conjunto."))

if es_degenerado(y_pred_profundo):
    print("\nAVISO: el modelo profundo predice una sola clase sobre prueba. Su F1-macro y su MCC")
    print("       no son comparables con los del clásico: no ha aprendido la tarea.")

print(f"\nReporte por clase del modelo profundo:\n")
print(classification_report(y_true_test, y_pred_profundo, target_names=NOMBRES_CLASE,
                            digits=4, zero_division=0))

### Matrices de confusión

Las cuatro celdas y lo que cuesta cada error en este problema:

- **falsa a falsa** — desinformación detectada.
- **real a real** — noticia verdadera reconocida.
- **falsa clasificada como real** — desinformación que pasa el filtro: el sistema no sirvió en ese caso.
- **real clasificada como falsa** — noticia verdadera marcada como falsa: censura de información legítima.

Los dos errores no cuestan lo mismo y esa asimetría hay que discutirla explícitamente en el informe.

In [ ]:
def dibujar_matriz(eje, y_true, y_pred, titulo):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    pct = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
    eje.imshow(pct, cmap="Blues", vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            eje.text(j, i, f"{cm[i, j]:,}\n{pct[i, j]:.1f}%", ha="center", va="center",
                     color="white" if pct[i, j] > 55 else "black", fontsize=10)
    eje.set_xticks([0, 1]); eje.set_xticklabels([f"pred: {n}" for n in NOMBRES_CLASE])
    eje.set_yticks([0, 1]); eje.set_yticklabels([f"real: {n}" for n in NOMBRES_CLASE])
    eje.set_title(titulo, fontsize=10)
    return cm


fig, ejes = plt.subplots(1, 2, figsize=(11, 4.4))
cm_clasico = dibujar_matriz(ejes[0], Y["test"], pred_clasico,
                            f"{CLASICOS[MEJOR_CLASICO]}\nacc {m_clasico['accuracy']:.4f}")
cm_profundo = dibujar_matriz(ejes[1], y_true_test, y_pred_profundo,
                             f"RoBERTa-BNE\nacc {m_profundo['accuracy']:.4f}")
plt.tight_layout()
plt.savefig(ruta("matrices_confusion.png"), dpi=150, bbox_inches="tight")
plt.show()

for etiqueta, cm in (("clásico", cm_clasico), ("profundo", cm_profundo)):
    print(f"{etiqueta:>9}: {cm[0, 1]:>4,} falsas que pasan el filtro · "
          f"{cm[1, 0]:>4,} reales marcadas como falsas")

pd.DataFrame(cm_profundo, index=[f"real_{n}" for n in NOMBRES_CLASE],
             columns=[f"pred_{n}" for n in NOMBRES_CLASE]).to_csv(ruta("matriz_confusion_profundo.csv"))

In [ ]:
concordancia = pd.DataFrame({
    "row_id": ID["test"],
    "y_true": Y["test"],
    "pred_clasico": pred_clasico,
    "pred_profundo": y_pred_profundo,
    "prob_falsa_profundo": probas_profundo[:, 0].round(4),
})
ok_clasico = concordancia["pred_clasico"] == concordancia["y_true"]
ok_profundo = concordancia["pred_profundo"] == concordancia["y_true"]
concordancia["categoria"] = np.select(
    [ok_clasico & ok_profundo, ~ok_clasico & ok_profundo, ok_clasico & ~ok_profundo],
    ["ambos_aciertan", "solo_profundo", "solo_clasico"], default="ambos_fallan")
concordancia = concordancia.merge(datos[["row_id", "texto"]], on="row_id", how="left")

tabla_conc = pd.DataFrame(
    [[int((ok_clasico & ok_profundo).sum()), int((ok_clasico & ~ok_profundo).sum())],
     [int((~ok_clasico & ok_profundo).sum()), int((~ok_clasico & ~ok_profundo).sum())]],
    index=["profundo acierta", "profundo falla"],
    columns=["clásico acierta", "clásico falla"])

seccion("CONCORDANCIA ENTRE LOS DOS MODELOS")
print(tabla_conc.to_string())
print("\nComo porcentaje del conjunto de prueba:\n")
print((tabla_conc / len(concordancia) * 100).round(2).to_string())

conteos = concordancia["categoria"].value_counts()
print(f"\n  Solo resuelve el profundo : {conteos.get('solo_profundo', 0):,}   "
      "(el valor que aporta la arquitectura)")
print(f"  Solo resuelve el clásico  : {conteos.get('solo_clasico', 0):,}")
print(f"  Fallan ambos              : {conteos.get('ambos_fallan', 0):,}   "
      "(núcleo duro para el análisis de fallas)")

duros = concordancia[concordancia["categoria"] == "ambos_fallan"]
if len(duros):
    print(f"\nMuestra de casos que fallan ambos modelos:\n")
    for _, r in duros.sample(n=min(5, len(duros)), random_state=SEMILLA_BASE).iterrows():
        print(f"  [row_id {int(r['row_id'])}] verdadera = {CONFIG['etiquetas'][int(r['y_true'])]}")
        print(f"    {str(r['texto'])[:210]}...\n")

concordancia.to_csv(ruta("concordancia_test.csv"), index=False)
print(f"Guardado 'concordancia_test.csv' en ./{DIR_SALIDA}/")

---
## §14 · Artefactos y resumen

Todo lo que un tercero necesita para reejecutar esta corrida y obtener lo mismo, o para escribir el
informe sin volver a encender la GPU:

| Archivo | Contenido |
|---|---|
| `configuracion.json` | Configuración completa, hiperparámetros efectivos, versiones, hardware y hash del archivo de datos |
| `particion.csv` | Fila por fila, a qué split fue cada noticia. Auditable |
| `resumen_particion.csv` | Conteos por split y etiqueta |
| `busqueda_clasicos.csv` | Rejilla completa de la línea base clásica sobre validación |
| `historial_mejor_semilla.csv` | Pérdida y métricas de validación por época |
| `resumen_semillas.csv` | Resultado de cada semilla, para la media y la desviación estándar |
| `metricas_finales.json` | Cifras reportables de ambos modelos con intervalos de confianza |
| `concordancia_test.csv` | Predicción de ambos modelos por noticia. Insumo del análisis de fallas |
| `terminos_indicativos.csv`, `exclusividad_terminos.csv` | Evidencia de la sonda del atajo |
| `matrices_confusion.png`, `curvas_entrenamiento.png` | Figuras para el informe |
| `modelo_profundo/` | Pesos y tokenizador del mejor punto de control |

In [ ]:
registro = {
    "descripcion": "Línea base clásica y ajuste fino de RoBERTa-BNE sobre la misma partición",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "configuracion": {k: v for k, v in CONFIG.items() if k != "alias_columnas"},
    "columnas_resueltas": COL,
    "datos": {
        "archivo": CONFIG["ruta_datos"],
        "md5": md5_archivo(CONFIG["ruta_datos"]),
        "informe_limpieza": informe_datos,
        "n_total": int(len(datos)),
        "n_train": int((datos["split"] == "train").sum()),
        "n_val": int((datos["split"] == "val").sum()),
        "n_test": int((datos["split"] == "test").sum()),
        "n_grupos": int(datos["grupo"].nunique()),
        "grupos_partidos": int(grupos_partidos),
        "proporciones": PROPORCIONES,
        "motivo_proporciones": motivo_prop,
        "pct_truncado": round(PCT_TRUNCADO, 2),
        "piso_trivial": round(PISO_TRIVIAL, 4),
    },
    "modelo_profundo": {
        "repo": MODELO, "commit": MODELO_SHA, "es_repo_del_articulo": ES_CANONICO,
        "repo_canonico": CONFIG["modelo_candidatos"][0],
        "hiperparametros_efectivos": HP,
    },
    "modelo_clasico": {
        "mejor": MEJOR_CLASICO,
        "configuracion": {k: (str(v) if isinstance(v, tuple) else v)
                          for k, v in mejores_clasicos[MEJOR_CLASICO].items()
                          if not k.startswith("val_")},
    },
    "entorno": {
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "sklearn": sklearn.__version__,
        "numpy": np.__version__, "pandas": pd.__version__,
        "dispositivo": str(DISPOSITIVO),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
}
with open(ruta("configuracion.json"), "w", encoding="utf-8") as f:
    json.dump(registro, f, indent=2, ensure_ascii=False, default=str)

resultados = {
    "n_test": int(len(y_true_test)),
    "piso_trivial": round(PISO_TRIVIAL, 4),
    "clasico": {"nombre": CLASICOS[MEJOR_CLASICO], "metricas": m_clasico,
                "ic95_f1_macro": [round(x, 4) for x in ic_clas]},
    "profundo": {"nombre": f"RoBERTa-BNE ({MODELO})", "metricas": m_profundo,
                 "ic95_f1_macro": [round(x, 4) for x in ic_prof],
                 "perdida_test": round(perdida_test, 4),
                 "semilla_reportada": mejor_corrida["semilla"],
                 "colapso": bool(mejor_corrida["colapso"]),
                 "por_semilla": resumen_semillas.to_dict("records"),
                 "media_test_f1_macro": round(float(resumen_semillas["test_f1_macro"].mean()), 4),
                 "std_test_f1_macro": round(float(resumen_semillas["test_f1_macro"].std(ddof=1))
                                            if len(resumen_semillas) > 1 else 0.0, 4)},
    "mcnemar": {"solo_clasico": b, "solo_profundo": c, "p_valor": round(p_valor, 4)},
    "matriz_confusion_profundo": cm_profundo.tolist(),
    "matriz_confusion_clasico": cm_clasico.tolist(),
}
with open(ruta("metricas_finales.json"), "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False, default=str)

if CONFIG["guardar_modelo"]:
    modelo_final = construir_modelo(HP, CONFIG)
    modelo_final.load_state_dict(mejor_estado)
    modelo_final.save_pretrained(ruta("modelo_profundo"))
    tokenizador.save_pretrained(ruta("modelo_profundo"))
    del modelo_final
    torch.cuda.empty_cache()
    print(f"Modelo guardado en ./{DIR_SALIDA}/modelo_profundo/")

print(f"\nArtefactos en ./{DIR_SALIDA}/:\n")
for nombre in sorted(os.listdir(DIR_SALIDA)):
    p = ruta(nombre)
    kb = (os.path.getsize(p) if os.path.isfile(p)
          else sum(os.path.getsize(os.path.join(dp, fn))
                   for dp, _, fs in os.walk(p) for fn in fs)) / 1024
    print(f"  {nombre:<34} {kb:>12,.1f} KB")

In [ ]:
desv = (resumen_semillas["test_f1_macro"].std(ddof=1) if len(resumen_semillas) > 1 else 0.0)

resumen = f"""
RESUMEN PARA EL INFORME
==============================================================================
DATOS
------------------------------------------------------------------------------
Archivo             : {CONFIG['ruta_datos']}
MD5                 : {registro['datos']['md5']}
Noticias            : {len(datos):,}   ({dict(datos[COL_ETIQUETA].value_counts().sort_index())})
Duplicados quitados : {informe_datos['duplicados_eliminados']:,}
Contradicciones     : {informe_datos['contradicciones_eliminadas']:,}
Fechas imposibles   : {informe_datos['fechas_imposibles']:,}
Truncadas a {HP['max_seq_length']} tok : {PCT_TRUNCADO:.2f}%

PARTICIÓN
------------------------------------------------------------------------------
Estrategia          : {CONFIG['estrategia_particion']}   ·   {motivo_prop}
Proporciones        : {PROPORCIONES[0]:.0%} / {PROPORCIONES[1]:.0%} / {PROPORCIONES[2]:.0%}
Tamaños             : train {registro['datos']['n_train']:,} · val {registro['datos']['n_val']:,} · test {registro['datos']['n_test']:,}
Grupos anti-fuga    : {registro['datos']['n_grupos']:,}   ·   grupos partidos: {grupos_partidos}
Piso trivial (test) : {PISO_TRIVIAL:.4f}

MODELO PROFUNDO
------------------------------------------------------------------------------
Repositorio         : {MODELO}  (commit {MODELO_SHA})
¿Repo del artículo? : {'sí' if ES_CANONICO else 'no, el canónico fue vaciado'}
Hiperparámetros     : {HP['origen']}
                      lr {HP['learning_rate']} · batch {HP['batch_size']} · máx {HP['epocas']} épocas
                      dropout {HP['dropout']} · weight decay {HP['weight_decay']} · warmup {HP['warmup_ratio']:.0%}
Semillas            : {CONFIG['semillas']}
Mejor época         : {mejor_corrida['mejor_epoca']} (semilla {mejor_corrida['semilla']})

RESULTADOS SOBRE PRUEBA (n = {len(y_true_test):,})
------------------------------------------------------------------------------
{CLASICOS[MEJOR_CLASICO]:<30} acc {m_clasico['accuracy']:.4f} · F1m {m_clasico['f1_macro']:.4f} · MCC {m_clasico['mcc']:.4f}
                               IC95% F1m [{ic_clas[0]:.4f}, {ic_clas[1]:.4f}]
{'RoBERTa-BNE':<30} acc {m_profundo['accuracy']:.4f} · F1m {m_profundo['f1_macro']:.4f} · MCC {m_profundo['mcc']:.4f}
                               IC95% F1m [{ic_prof[0]:.4f}, {ic_prof[1]:.4f}]
                               entre semillas: {resumen_semillas['test_f1_macro'].mean():.4f} ± {desv:.4f}

Mejora del profundo sobre el clásico : {(m_profundo['f1_macro'] - m_clasico['f1_macro']) * 100:+.2f} puntos de F1-macro
Mejora del profundo sobre el piso    : {(m_profundo['accuracy'] - PISO_TRIVIAL) * 100:+.2f} puntos de accuracy
McNemar                              : p = {p_valor:.4f} ({'significativo' if p_valor < 0.05 else 'no significativo'})

ENTORNO
------------------------------------------------------------------------------
python {registro['entorno']['python']} · torch {registro['entorno']['torch']} · transformers {registro['entorno']['transformers']} · sklearn {registro['entorno']['sklearn']}
{registro['entorno']['dispositivo']} ({registro['entorno']['gpu']})
==============================================================================
"""
print(resumen)
with open(ruta("resumen_informe.txt"), "w", encoding="utf-8") as f:
    f.write(resumen)
print(f"Guardado en ./{DIR_SALIDA}/resumen_informe.txt")